# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 6: Interactive Map with Folium

**Author:** Roberto Cortez  
**GitHub:** rtez-tech

We build an interactive map to visualize launch sites, color-coded launch outcomes
(clustered markers), and the distance from a launch site to nearby landmarks
(coastline, railway, highway, city).

> **Note for the presentation:** Folium maps are interactive HTML. Run each map cell,
> then take screenshots for slides 35–37.

In [1]:
import folium
import pandas as pd
from math import sin, cos, sqrt, atan2, radians
from folium.plugins import MarkerCluster
from folium.features import DivIcon

ModuleNotFoundError: No module named 'folium'

In [2]:
spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv")
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site','Lat','Long']]
launch_sites_df

NameError: name 'pd' is not defined

### Task 1 — Mark all launch sites on a map

In [3]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for idx, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    folium.Circle(coordinate, radius=1000, color='#d35400', fill=True).add_child(
        folium.Popup(row['Launch Site']))
    folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size:12px;color:#d35400;"><b>%s</b></div>' % row['Launch Site'])
        ).add_to(site_map)
    folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_to(site_map)
site_map

NameError: name 'folium' is not defined

### Task 2 — Color-coded launch outcomes (marker cluster)
Green = success (class 1), Red = failure (class 0).

In [4]:
marker_cluster = MarkerCluster()
def assign_color(launch_class):
    return 'green' if launch_class == 1 else 'red'
spacex_df['marker_color'] = spacex_df['class'].apply(assign_color)

site_map.add_child(marker_cluster)
for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        [record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']))
    marker_cluster.add_child(marker)
site_map

NameError: name 'MarkerCluster' is not defined

### Task 3 — Distance from a launch site to its proximities

In [5]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1; dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Example: CCAFS SLC-40 to the nearest coastline
launch_site_lat, launch_site_lon = 28.563197, -80.576820
coastline_lat, coastline_lon = 28.56367, -80.57163
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon,
                                        coastline_lat, coastline_lon)
print("Distance to coastline: {:.2f} km".format(distance_coastline))

distance_marker = folium.Marker([coastline_lat, coastline_lon],
    icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size:12px;color:#0000ff;"><b>%.2f KM</b></div>' % distance_coastline))
site_map.add_child(distance_marker)
folium.PolyLine([[launch_site_lat,launch_site_lon],[coastline_lat,coastline_lon]],
                weight=1).add_to(site_map)
site_map

NameError: name 'radians' is not defined

### Findings (write your own observations from the rendered maps)
- Launch sites are all near coastlines (safety: debris falls into the ocean).
- Sites are close to railways and highways (logistics) but kept away from dense cities.
- Florida sites (CCAFS, KSC) cluster on the east coast; VAFB is on the California coast.

**Next:** Lab 7 — Dashboard with Plotly Dash.